In [1]:
# Importar as bibliotecas

from datetime import date
import json
from pathlib import Path
import os
import sys

In [2]:
# Resolver os 'imports' do projeto 

PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

from utils.classes.pandas_dataframe import PandasDataframe as pd_dataframe

In [3]:
# Traz as opções de configuração do JSON

json_file = os.path.abspath('../../../../options.json')

with open(json_file, 'r', encoding='utf-8') as j_file:
    json_data = json.load(j_file)

In [4]:
# Declaração dos caminhos

cdi_data = os.path.abspath('../../../../data')
cdi_metrics = os.path.join(cdi_data, 'metrics')

cdi_dir = os.path.join(cdi_metrics, 'cdi')
cdi_file = 'cdi_data.csv'
cdi_path = os.path.join(cdi_dir, cdi_file)

In [5]:
# Obtenção da planilha com os dados diários do CDI

df_cdi = pd_dataframe(cdi_path, None)
df_cdi.csv_to_df()

In [6]:
# Settar dados de início e fim de valorização

start_date = json_data['METRICS']['CDI'].get("START_DATE", False)
end_date   = json_data['METRICS']['CDI'].get("END_DATE", False)

In [7]:
# Ajuste das datas (início e fim) de CDI caso não descritas 

date_start_object = {
    'is_valid': False,
    'start_date': '',
}

date_end_object = {
     'is_valid': False,
     'end_date': '',
}

if start_date == False:
    date_start_object['is_valid'] = True
    date_start_object['start_date'] = df_cdi.df.iloc[0]['DATE']

if end_date == False:
    date_end_object['is_valid'] = True
    date_end_object['end_date'] = df_cdi.df.iloc[-1]['DATE']

if date_start_object['is_valid'] == True or date_end_object['is_valid'] == True:
        
    file_content = ''
    with open(json_file, 'r', encoding='utf8') as j_file:
        
        file_content = json.load(j_file)

        if date_start_object['is_valid'] == True:
            file_content['METRICS']['CDI']['START_DATE'] = date_start_object['start_date']    
            start_date = date_start_object['start_date']          

        if date_end_object['is_valid'] == True:
            file_content['METRICS']['CDI']['SOLD_DATE'] = date_end_object['end_date']
            end_date   = date_end_object['end_date']

    with open(json_file, 'w', encoding='utf8') as j_file:
        json.dump(file_content, j_file, indent=4, ensure_ascii=False)

In [8]:
# Filtragem pela data dos dados

COLUMN_NAME = 'DATE'
df_cdi.query_date(start_date, end_date, COLUMN_NAME)

In [9]:
# Criar uma lista com os dados de valorização

list_cdi_daily = (df_cdi.df['CDI_TAX']).to_list()
list_day_date = (df_cdi.df['DATE']).to_list()
list_cdi_daily.pop() # Valorização começa em 0

first_cdi_valuation = 0
list_cdi_valuation = [first_cdi_valuation]
last_cdi_tax_calc = 1

for cdi_tax in list_cdi_daily:
    last_cdi_tax_calc =  (1 + (cdi_tax / 100)) * last_cdi_tax_calc
    list_cdi_valuation.append((last_cdi_tax_calc - 1) * 100)

In [10]:
# Geração de um '.csv' com a valorização do CDI

DICT_CDI_VALUATION = { 'METRIC_NAME': 'CDI', 'VALUATION_PERCENT': list_cdi_valuation, 'DATE': list_day_date }

df_cdi_valuation = pd_dataframe(None, None, dict=DICT_CDI_VALUATION)
df_cdi_valuation.dict_to_df()

cdi_valuation_archive = 'cdi_valuation.csv'
cdi_valuation_path = os.path.join(cdi_dir, cdi_valuation_archive)

df_cdi_valuation.df_to_csv(cdi_valuation_path)